  <center><p float="center">
  
  </p></center>

  <center><font size=10>Generative AI for Business Applications</center></font>


<center><p float="center">
  <img src="https://blog.slantco.com/wp-content/uploads/2025/09/top-restaurant-review-platforms-middle-east-1024x682.jpg" width=720/>
</p></center>

<center><font size=6>Restaurant Review Analysis</center></font>

### **Problem Statement**

A food aggregator wants to understand customer experiences from restaurant reviews. Manually analyzing a large number of reviews is time-consuming, so an automated system is needed to identify customer sentiment and important feedback.

### **Business Context**

Customer satisfaction is important for the success of restaurants and the overall food-aggregator brand. Customer reviews provide valuable information about food quality, service, ambience, and overall satisfaction.

### **Objective**

The objective is to develop an LLM-based sentiment analysis system using prompt engineering to:

* Identify Positive, Negative, and Neutral sentiment.
* Analyze sentiment for food quality, service, and ambience.
* Identify liked and disliked features.
* Help restaurants understand feedback and improve customer satisfaction.

### **Data Dictionary**

The dataset comprises three columns:

   **restaurant\_id** – Unique identifier for each restaurant.

   **rating\_review** – Numerical or categorical rating provided by the customer.

   **review\_full** – Full text of the customer’s review.
   

   The dataset contains **20 rows and 3 columns**.

   There are **no missing values** in the dataset.


### **Expected Outcome**

The system is expected to provide:

* Overall sentiment for each review.
* Aspect-level sentiment for food, service, and ambience.
* Liked and disliked features of each aspect.
* Useful restaurant-specific insights that can help improve service quality and customer satisfaction.
* Personalized responses to customers based on their reviews.

## **Installing and Importing Necessary Libraries**

In [ ]:
!pip install -q transformers==4.53.2 \
                  accelerate==1.8.1 \
                  bitsandbytes==0.46.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 49.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


# **Interpretation:**
Imports the required libraries for data handling, JSON processing, system operations, and using a language model for sentiment analysis.

In [ ]:
import pandas as pd
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

## **Import the dataset**



<font size=3 ><b> Mount the Google Drive
</font>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive




<font size=3 ><b> Load the CSV file named "restaurant_reviews.csv" and store it in the variable data.
</font>

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/GenAI/Dataset/Restaurant_Review_Analysis_Notebook.ipynb")

## **Data Overview**



<font size=3 ><b> Display the first 5 rows of the data.
</font>

In [ ]:

data.head()

,"{""cells"":[{""cell_type"":""markdown""","id:""lQMN7B5FKR2S""","metadata:{""id"":""lQMN7B5FKR2S""}","source:[""<center><p float=\""center\"">\n""",\n,</p></center>\n,\n,<center><font size=10>Generative AI for Business Applications</center></font>\n,<center><font size=6>Large Language Models & Prompt Engineering - Week 2</center></font>]},"{""cell_type"":""markdown""",...,"state:{""_model_module"":""@jupyter-widgets/controls"".32","_model_module_version:""1.5.0"".76","_model_name:""DescriptionStyleModel"".21",_view_count:null.120,"_view_module:""@jupyter-widgets/base"".76","_view_module_version:""1.2.0"".76","_view_name:""StyleView"".32","description_width:""""}}}}}}",nbformat:4,nbformat_minor:5}




<font size=3 ><b> Display the number of rows and columns in the data.
</font>

In [ ]:
data.shape

(0, 6788)

**Observations**

- Data has 20 rows and 3 columns

In [ ]:

data.isnull().sum()

,0
"{""cells"":[{""cell_type"":""markdown""",0
"id:""lQMN7B5FKR2S""",0
"metadata:{""id"":""lQMN7B5FKR2S""}",0
"source:[""<center><p float=\""center\"">\n""",0
\n,0
...,...
"_view_module_version:""1.2.0"".76",0
"_view_name:""StyleView"".32",0
"description_width:""""}}}}}}",0
nbformat:4,0


**Observations**

- There are no missing values in the data

In [ ]:

df = data.copy()

# **Model Loading**

In [ ]:
file_name = '/content/drive/MyDrive/GenAI/Dataset/config.json'
with open(file_name, 'r') as file:
    config = json.load(file)
    HF_TOKEN = config.get("HF_TOKEN")


# Store API credentials in environment variables
os.environ['HF_TOKEN'] = HF_TOKEN




<font size=3 ><b> Load the mistralai/Mistral-7B-Instruct-v0.1 from hugging face using 8-bit quantization.

</font>

In [ ]:
import torch

model_id = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_8bit=True,  # Load the model with 8-bit quantization
    torch_dtype=torch.float16,         # Use 16-bit floats on GPU
    device_map="auto",                 # Automatically assign GPU or CPU
    token=HF_TOKEN
)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

* load_in_8bit=True: Loads the model using 8-bit quantization to save memory.
* torch_dtype=torch.float16: Uses half-precision (16-bit) floats for faster computation on GPU.
* device_map="auto": Automatically places model layers across available devices.


Hugging Face model is now ready. Let’s test it on an example input.



<font size=3 ><b> Ask the Mistral model: What is the capital of France?
</font>

In [ ]:

prompt = "### Question: What is the capital of France?\n### Answer:"


inputs = tokenizer(prompt, return_tensors="pt").to(model.device)


outputs = model.generate(**inputs)


response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


### Question: What is the capital of France?
### Answer: Paris


Now that the model is returning results successfully.



<font size=3 ><b> Create a function that accepts a prompt and query, and returns the response generated by the Mistral model.
</font>

In [ ]:
def query_mistral(prompt, query):
    """
    Queries the Mistral model with a given prompt and query.

    Args:
        prompt (str): The prompt for the model.
        query (str): The query to be answered by the model.

    Returns:
        str: The model's response.
    """
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": query}
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)

    pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    attention_mask = (inputs != pad_token_id).long()

    outputs = model.generate(
        inputs,
        attention_mask=attention_mask,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=pad_token_id
    )


    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    return response


# **Reviews Sentiment Analysis**

## **1. Overall Sentiment Analysis**

In [ ]:
# defining the instructions for the model
instruction_1 = """
    You are an AI analyzing restaurant reviews. Classify the sentiment of the provided review into the following categories:
    - Positive
    - Negative
    - Neutral

    And return in only JSON format. No extra text and analysis
    {"Sentiment":"Positive"}
"""



<font size=3 ><b> Define a function named classify_sentiment that takes the instructtion_1 and the review text as input, gets the result from query_mistral function, and returns the result in a JSON format.
</font>

In [ ]:
def classify_sentiment(prompt, query):
    try:
        response_text = query_mistral(prompt, query)
        # Attempt to parse the response text as JSON
        classification_result = json.loads(response_text)
        return classification_result
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from OpenAI response: {e}")
        print(f"Raw OpenAI response: {response_text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None





<font size=3 ><b>Generate the category for each support_ticket_text in the DataFrame using the classify_ticket_openai function, and store the result in a new column.

</font>

In [ ]:
# Apply the classification function to each row in the DataFrame
df['Sentiment'] = df['review_full'].apply(lambda x: classify_sentiment(instruction_1, x)['Sentiment'] if classify_sentiment(instruction_1, x) else None)

In [ ]:
df

,restaurant_ID,rating_review,review_full,Sentiment
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive
4,EAT456,5,Had a great experience in the restaurant food ...,Positive
5,RST-A1,5,We came across Perch by accident and had dinne...,Positive
6,DSH404,5,We went there on birthday special time. A nice...,Negative
7,GRT505,3,Our visit to Green Bites on a busy Saturday ev...,Neutral
8,MMM606,3,"At Bella Cuisine, the cozy atmosphere and frie...",Neutral
9,FST707,3,Our dinner at Spice Haven provided a neutral e...,Neutral


In [ ]:
df['Sentiment'].value_counts()

,count
Sentiment,
Negative,8
Positive,6
Neutral,6


Across the different restaurants, negative sentiment slightly outweighs positive and neutral feedback, indicating more dissatisfaction overall.


## **2. Sentiment toward Different Aspects of the Experience**

In [ ]:
# defining the instructions for the model
instruction_2 = """
    You are an AI analyzing restaurant reviews. Classify the following aspects in the review and classify the sentiment of each aspect as "Positive", "Negative", or "Neutral":
    1. "Food Quality"
    2. "Service"
    3. "Ambience"

    Output the overall sentiment and sentiment for each category in a JSON format with the following keys:
    {
        "Food Quality": "your_sentiment_prediction",
        "Service": "your_sentiment_prediction",
        "Ambience": "your_sentiment_prediction"
    }

    In case one of the three aspects is not mentioned in the review, set "Not Applicable" (including quotes) for the corresponding JSON key value.
    Only return the JSON, do not return any other information.
"""



<font size=3 ><b>Define a function that takes the instruction_2 prompt and query as input get the result from query_mistral function return the result in JSON format
</font>

In [ ]:
def classify_aspect_sentiment(prompt, query):
    """
    Classifies the sentiment of aspects of the review using the Mistral model
    and returns the result as a JSON object.

    Args:
        prompt (str): The prompt for the model (instruction_2).
        query (str): The review text.

    Returns:
        dict: A dictionary containing the sentiment classification for each aspect,
              or None if JSON decoding fails.
    """
    try:
        response_text = query_mistral(prompt, query)
        # Attempt to parse the response text as JSON
        classification_result = json.loads(response_text)
        return classification_result
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from model response: {e}")
        print(f"Raw model response: {response_text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None



<font size=3 ><b>Generate the aspect_sentiment for each review in the DataFrame using classify_aspect_sentiment, store it in a new column, and extract individual fields into separate columns.
</font>

In [ ]:
# Apply the classification function to each row in the DataFrame
df['aspect_sentiment'] = df['review_full'].apply(lambda x: classify_aspect_sentiment(instruction_2, x))

# Normalize the JSON results into separate columns
aspect_sentiment_df = pd.json_normalize(df['aspect_sentiment'])

# Concatenate the original DataFrame with the new columns
df = pd.concat([df, aspect_sentiment_df], axis=1)

In [ ]:
df

,restaurant_ID,rating_review,review_full,Sentiment,aspect_sentiment,Food Quality,Service,Ambience
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,"{'Food Quality': 'Positive', 'Service': 'Not A...",Positive,Not Applicable,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Not Applicable
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive
5,RST-A1,5,We came across Perch by accident and had dinne...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive
6,DSH404,5,We went there on birthday special time. A nice...,Negative,"{'Food Quality': 'Negative', 'Service': 'Neutr...",Negative,Neutral,Positive
7,GRT505,3,Our visit to Green Bites on a busy Saturday ev...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Negati...",Neutral,Negative,Neutral
8,MMM606,3,"At Bella Cuisine, the cozy atmosphere and frie...",Neutral,"{'Food Quality': 'Negative', 'Service': 'Negat...",Negative,Negative,Positive
9,FST707,3,Our dinner at Spice Haven provided a neutral e...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Not Applicable


In [ ]:
df['Food Quality'].value_counts()

,count
Food Quality,
Positive,7
Negative,7
Neutral,5
Not Applicable,1


Overall, food quality feedback leans negative, with 8 unfavorable mentions compared to 6 positive, while a few reviews were neutral or not applicable.


In [ ]:
df['Service'].value_counts()

,count
Service,
Negative,10
Positive,8
Not Applicable,1
Neutral,1


Service-related feedback is predominantly negative, with 10 unfavorable mentions outweighing the 8 positive and 2 neutral reviews.


In [ ]:
df['Ambience'].value_counts()

,count
Ambience,
Positive,9
Neutral,5
Not Applicable,4
Negative,2


Ambience is viewed largely positively, with 8 favorable mentions and minimal negative feedback.


## **3. Identifying Liked/Disliked Features of the Different Aspects of the Experience**

In [ ]:
# defining the instructions for the model
instruction_3 = """

You are an AI model assigned to analyze restaurant reviews. Your task is to extract the **specific features** that the customer **liked or disliked**, categorized under the following aspects of the dining experience:

* Food Quality
* Service
* Ambience

Return the result in the following strict JSON format:

{
  "Food Quality Features": ["specific liked/disliked features"],
  "Service Features": ["specific liked/disliked features"],
  "Ambience Features": ["specific liked/disliked features"]
}

**Instructions:**

* Only list **concrete features** (e.g., “taste”, “temperature”, “presentation”, “waiting time”, “staff behavior”, “lighting”, “music volume”) that are mentioned positively or negatively in the review.
* Do **not** include generic phrases like “liked feature” or “disliked feature”.
* If a particular aspect has no feature mentioned, return an empty list for that aspect.
* Output **only the JSON**, with keys exactly as specified. Do not add any explanations or comments.

"""



<font size=3 ><b>Define a function that takes the instruction_3 prompt and query as input get the result from query_mistral function return the result in JSON format
</font>

In [ ]:
def classify_features(prompt, query):
    """
    Extracts liked/disliked features from the review using the Mistral model
    and returns the result as a JSON object.

    Args:
        prompt (str): The prompt for the model (instruction_3).
        query (str): The review text.

    Returns:
        dict: A dictionary containing the extracted features for each aspect,
              or None if JSON decoding fails.
    """
    try:
        response_text = query_mistral(prompt, query)
        # Attempt to parse the response text as JSON
        classification_result = json.loads(response_text)
        return classification_result
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from model response: {e}")
        print(f"Raw model response: {response_text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None



<font size=3><b>Generate the Features for each review in the DataFrame using classify_features, store them in a new column, and extract individual fields into separate columns.
</font>

In [ ]:
# Apply the classification function to each row in the DataFrame
df['aspect_features'] = df['review_full'].apply(lambda x: classify_features(instruction_3, x))

# Normalize the JSON results into separate columns
aspect_features_df = pd.json_normalize(df['aspect_features'])

# Concatenate the original DataFrame with the new columns
df = pd.concat([df, aspect_features_df], axis=1)

In [ ]:
df

,restaurant_ID,rating_review,review_full,Sentiment,aspect_sentiment,Food Quality,Service,Ambience,aspect_features,Food Quality Features,Service Features,Ambience Features
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['pizza', 'hummus', ...","[pizza, hummus, pita bread]","[mask on staff, good sanitisation]","[outdoor and indoor interior, quaint and cute,..."
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,"{'Food Quality': 'Positive', 'Service': 'Not A...",Positive,Not Applicable,Positive,"{'Food Quality Features': ['Margherita pizza',...","[Margherita pizza, Hummus, Pita, Fussili Alfre...",[],"[peaceful, plants]"
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['taste'], 'Service ...",[taste],"[service, Subham Barnwal, great]",[decorum]
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Not Applicable,"{'Food Quality Features': [], 'Service Feature...",[],"[Ms. Laxmi, handling client needs, specialty, ...",[]
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['taste', 'fabulous'...","[taste, fabulous, presentation, enjoyed meal]","[staff was nice, professional, helpful, comfor...","[wonderful experience, staff made it more comf..."
5,RST-A1,5,We came across Perch by accident and had dinne...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['taste', 'presentat...","[taste, presentation, menu, replicate dishes]","[attentive, best, range of drinks]","[stylish, simple, lively, tasteful music]"
6,DSH404,5,We went there on birthday special time. A nice...,Negative,"{'Food Quality': 'Negative', 'Service': 'Neutr...",Negative,Neutral,Positive,{'Food Quality Features': ['Pulled Duck Salad'...,"[Pulled Duck Salad, taste, marination, value-f...",[],"[interior, friendly, not intimidating, soft li..."
7,GRT505,3,Our visit to Green Bites on a busy Saturday ev...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Negati...",Neutral,Negative,Neutral,"{'Food Quality Features': ['taste', 'menu opti...","[taste, menu options, price]","[slow service, attentive staff]",[busy atmosphere]
8,MMM606,3,"At Bella Cuisine, the cozy atmosphere and frie...",Neutral,"{'Food Quality': 'Negative', 'Service': 'Negat...",Negative,Negative,Positive,"{'Food Quality Features': ['flavorful', 'disti...","[flavorful, distinctive taste]",[inconsistent],"[cozy atmosphere, friendly staff]"
9,FST707,3,Our dinner at Spice Haven provided a neutral e...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Not Applicable,"{'Food Quality Features': ['menu options', 'qu...","[menu options, quality of food, preparation]","[efficiency, attentiveness, staff members]",[]


## **4. Sharing a Response**

In [ ]:
# defining the instructions for the model
instruction_4 = """
You are an AI analyzing restaurant reviews. Your task is to generate a **polite and empathetic response** directly based on the sentiment of the review.

Follow this structure:

* Start with a thank you for their feedback.
* Then:

  1. If the review is positive, say you’re glad they enjoyed the experience and that it would be great to have them again.
  2. If the review is neutral, thank them and ask what the restaurant could have done better.
  3. If the review is negative, apologize for the inconvenience and mention that the team will look into the concerns raised.

Constraints:

* Do not start with “Dear Customer” or any greeting.
* Only output the final response. No sentiment label, explanation, or extra text.
"""



<font size=3 ><b>Define a function that takes the instruction_4 prompt and query as input, gets the result from the query_mistral function, and returns the result
</font>

In [ ]:
def generate_customer_response(prompt, query):
    """
    Generates a customer response based on the review using the Mistral model.

    Args:
        prompt (str): The prompt for the model (instruction_4).
        query (str): The review text.

    Returns:
        str: The generated customer response.
    """
    response_text = query_mistral(prompt, query)
    return response_text



<font size=3 ><b>Generate the response for each review in the DataFrame using generate_customer_response, and store them in a new column.
</font>

In [ ]:
# Apply the classification function to each row in the DataFrame
df['customer_response'] = df['review_full'].apply(lambda x: generate_customer_response(instruction_4, x))

In [ ]:
df

## **Conclusions**

The project successfully uses an LLM-based approach to analyze restaurant reviews, identify overall and aspect-level sentiments, and extract useful customer feedback. These insights can help restaurants understand customer satisfaction and improve their services

# **Future Improvements**

* Increase the dataset size with more restaurant reviews.
* Support multiple languages for review analysis.
* Improve aspect-level sentiment analysis.
* Provide more detailed restaurant-specific insights.
* Develop a real-time review analysis system.
* Generate automated responses to customer reviews.